# Checkpoint 2: Non-Linear Classical Models & EEGNet
**Course:** Applications of Soft Computing (ASC), 2026–2027  
**Track:** Track A — EEG Seizure Prediction  
**Milestone:** Checkpoint 2 — Classical Models, EEGNet, Paired Bootstrap Comparison

---

## Scientific Rationale

Checkpoint 1 established that a **Logistic Regression** baseline on 180 neurophysiological biomarkers
achieves AUC ≈ 0.997 with 100% seizure sensitivity and 0 false alarms per hour on patient `chb01`.

Checkpoint 2 answers two key questions:

1. **Do non-linear classical models (Random Forest, SVM, MLP) outperform the linear baseline** on the biomarker representation?
2. **Does learning directly from raw EEG windows via EEGNet** (a compact CNN) offer competitive performance without hand-crafted features?

All models are evaluated under the same **Leave-One-Seizure-Out (LOSO)** protocol to prevent data leakage,
and the baseline comparison uses **paired bootstrap confidence intervals on ΔAUC** to quantify statistical significance.

### References
- Lawhern et al. (2018), *EEGNet*, J Neural Eng ([DOI: 10.1088/1741-2552/aace8c](https://doi.org/10.1088/1741-2552/aace8c))
- Kuhlmann et al. (2018), Nat Rev Neurol ([DOI: 10.1038/s41582-018-0055-2](https://doi.org/10.1038/s41582-018-0055-2))
- Ren et al. (2022), Front Neurol ([DOI: 10.3389/fneur.2022.1016224](https://doi.org/10.3389/fneur.2022.1016224))
- Breiman (2001), *Random Forests*, Mach Learn ([DOI: 10.1023/A:1010933404324](https://doi.org/10.1023/A:1010933404324))
- Ibrahim et al. (2023), Front Signal Process ([DOI: 10.3389/frsip.2023.1175305](https://doi.org/10.3389/frsip.2023.1175305))


In [ ]:
# Standard imports and path setup
import sys
import json
import time
import logging
import warnings
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s')

# Add project root to path
PROJECT_ROOT = Path('.').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")

In [ ]:
# Import project modules
from src.config import default_config, RAW_DATA_DIR, FIGURES_DIR, TABLES_DIR
from src.data.dataset import SeizurePredictionDataset
from src.features.biomarkers import BiomarkerExtractor
from src.models.baseline import LogisticRegressionBaseline, RandomChancePredictor
from src.models.classical import (
    RandomForestSeizurePredictor,
    SVMSeizurePredictor,
    MLPSeizurePredictor
)
from src.models.eegnet import EEGNetTrainer
from src.evaluation.loso import run_loso_cv, compare_models_bootstrap
from src.visualization.plots import plot_roc_and_pr_curves
from src.visualization.plots_cp2 import (
    plot_learning_curves,
    plot_model_comparison_bars,
    plot_bootstrap_ci,
    plot_feature_importance,
    plot_confusion_matrices,
    plot_alarm_timeline
)

PATIENT_ID = 'chb01'
default_config.ensure_directories()
print('All modules imported successfully.')

---
## 1. Data Loading & Window Extraction

We reuse the Checkpoint 1 preprocessing pipeline:
- **Bandpass filter:** 0.5–45 Hz (removes DC drift and high-frequency noise)
- **Notch filter:** 60 Hz (powerline interference)
- **Resample:** 256 Hz → 128 Hz (halves parameter count, preserves gamma band)
- **Windowing:** 10-second overlapping windows with 50% preictal overlap
- **Labelling:** Preictal = [onset − 35 min, onset − 5 min]; Interictal = ≥ 4h from any seizure

In [ ]:
dataset = SeizurePredictionDataset(
    patient_id=PATIENT_ID,
    raw_dir=RAW_DATA_DIR,
    config=default_config
)
patient_data = dataset.build_patient_dataset()

X_raw       = patient_data['X']           # (N, channels, samples) — raw EEG
y           = patient_data['y']           # (N,) — 1=preictal, 0=interictal
seizure_ids = patient_data['seizure_ids']

print(f"Dataset shape:      {X_raw.shape}")
print(f"Preictal windows:   {(y==1).sum()}")
print(f"Interictal windows: {(y==0).sum()}")
print(f"Seizures:           {len(set(s for s in seizure_ids if s != 'interictal'))}")

---
## 2. Representation 1: Biomarker Feature Extraction (180 features)

For each 10-second window and each of 18 bipolar EEG channels, we compute:

| Feature | Channels | Source |
|---------|----------|--------|
| Relative band power (δ, θ, α, β, γ) via Welch PSD | 18 × 5 = 90 | Jiang et al. (2023) |
| Slow-to-fast power ratio (δ+θ)/(α+β) | 18 × 1 = 18 | Kuhlmann et al. (2018) |
| Normalised spectral entropy | 18 × 1 = 18 | Alvarado-Rojas et al. (2014) |
| Hjorth parameters (Activity, Mobility, Complexity) | 18 × 3 = 54 | Jiang et al. (2023) |
| **Total** | **180** | |

In [ ]:
extractor = BiomarkerExtractor(config=default_config)
X_features = extractor.extract_dataset_features(X_raw)
print(f"Feature matrix: {X_features.shape}  (windows × biomarkers)")

# Build feature names for interpretability
channels = default_config.channels
band_names = list(default_config.frequency_bands.keys())
feature_names = []
for ch in channels:
    for b in band_names:
        feature_names.append(f"{ch}_rel_{b}")
    feature_names.append(f"{ch}_slow_fast_ratio")
    feature_names.append(f"{ch}_spec_entropy")
    feature_names.append(f"{ch}_hjorth_activity")
    feature_names.append(f"{ch}_hjorth_mobility")
    feature_names.append(f"{ch}_hjorth_complexity")

print(f"Feature names generated: {len(feature_names)}")

---
## 3. LOSO Cross-Validation: Biomarker Models

Each model is trained and evaluated under **Leave-One-Seizure-Out** cross-validation:
- K folds = K distinct seizures in the dataset
- Each fold: hold out one complete preictal seizure block + proportional interictal slice
- No preictal windows are shuffled between folds (prevents temporal autocorrelation leakage)

### 3.1 Logistic Regression Baseline (carried from Checkpoint 1)

In [ ]:
lr_model = LogisticRegressionBaseline(C=1.0, random_state=42)
lr_result = run_loso_cv(X_features, y, seizure_ids, lr_model, config=default_config)

agg = lr_result['aggregated']
print(f"Logistic Regression Baseline:")
print(f"  AUC-ROC:  {agg['roc_auc_mean']:.4f} ± {agg['roc_auc_std']:.4f}")
print(f"  Sensitivity: {agg['seizure_sensitivity_mean']*100:.1f}%")
print(f"  FPR/h:    {agg['fpr_per_hour_mean']:.4f}")

### 3.2 Random Forest

A Random Forest with 300 trees learns non-linear decision boundaries in the 180-dimensional biomarker space.
Unlike Logistic Regression, it can model interactions between spectral bands across channels.
The `balanced` class weight automatically adjusts for the preictal/interictal imbalance.

Reference: Breiman (2001). Random Forests. *Machine Learning*, 45(1). [DOI: 10.1023/A:1010933404324](https://doi.org/10.1023/A:1010933404324)

In [ ]:
rf_model = RandomForestSeizurePredictor(n_estimators=300, random_state=42)
rf_result = run_loso_cv(X_features, y, seizure_ids, rf_model, config=default_config)

agg = rf_result['aggregated']
print(f"Random Forest (n=300 trees):")
print(f"  AUC-ROC:  {agg['roc_auc_mean']:.4f} ± {agg['roc_auc_std']:.4f}")
print(f"  Sensitivity: {agg['seizure_sensitivity_mean']*100:.1f}%")
print(f"  FPR/h:    {agg['fpr_per_hour_mean']:.4f}")

### 3.3 Support Vector Machine (RBF Kernel)

An SVM with an RBF kernel maps the biomarker features into a high-dimensional space
where a maximum-margin hyperplane separates preictal and interictal classes.
Platt scaling provides calibrated probability estimates needed for ROC/PR computation.

Reference: Chang & Lin (2011). LIBSVM. *ACM TIST*, 2(3). [DOI: 10.1145/1961189.1961199](https://doi.org/10.1145/1961189.1961199)

In [ ]:
svm_model = SVMSeizurePredictor(C=1.0, gamma='scale', random_state=42)
svm_result = run_loso_cv(X_features, y, seizure_ids, svm_model, config=default_config)

agg = svm_result['aggregated']
print(f"SVM (RBF, C=1.0, gamma='scale', Platt-calibrated):")
print(f"  AUC-ROC:  {agg['roc_auc_mean']:.4f} ± {agg['roc_auc_std']:.4f}")
print(f"  Sensitivity: {agg['seizure_sensitivity_mean']*100:.1f}%")
print(f"  FPR/h:    {agg['fpr_per_hour_mean']:.4f}")

### 3.4 Multilayer Perceptron (Shallow ANN)

A shallow MLP with two hidden layers (256 → 128) trained with Adam and early stopping.
L2 regularisation (α=1e-3) and a 15% validation fraction prevent overfitting on the small preictal pool.

Reference: Jiang et al. (2023). Front Neurosci, 17, 1191683. [DOI: 10.3389/fnins.2023.1191683](https://doi.org/10.3389/fnins.2023.1191683)

In [ ]:
mlp_model = MLPSeizurePredictor(hidden_layer_sizes=(256, 128), alpha=1e-3, max_iter=500, random_state=42)
mlp_result = run_loso_cv(X_features, y, seizure_ids, mlp_model, config=default_config)

agg = mlp_result['aggregated']
print(f"MLP [256→128 ReLU, Adam, early stopping]:")
print(f"  AUC-ROC:  {agg['roc_auc_mean']:.4f} ± {agg['roc_auc_std']:.4f}")
print(f"  Sensitivity: {agg['seizure_sensitivity_mean']*100:.1f}%")
print(f"  FPR/h:    {agg['fpr_per_hour_mean']:.4f}")

---
## 4. Representation 2: EEGNet (Raw EEG)

EEGNet learns temporal and spatial filters directly from the raw filtered EEG,
bypassing manual biomarker engineering.

**Architecture** (Lawhern et al. 2018):
```
Input:  (batch, 1, 18_channels, 1280_samples)
Block1: TemporalConv(F1=8, k=64) → DepthwiseConv(D=2) → BN → ELU → AvgPool(4) → Drop(0.5)
Block2: SeparableConv(F2=16, k=16) → BN → ELU → AvgPool(8) → Drop(0.5)
Head:   Flatten → Linear(n_classes=2)
```
- **MaxNorm constraints** on depthwise weights (max=1) and classifier (max=0.25)
- **Adam optimiser** with cosine annealing LR schedule
- **Early stopping** on validation AUC (patience=20 epochs)

Reference: Lawhern et al. (2018). EEGNet. *J Neural Eng*, 15(5). [DOI: 10.1088/1741-2552/aace8c](https://doi.org/10.1088/1741-2552/aace8c)

In [ ]:
try:
    import torch
    print(f"PyTorch version: {torch.__version__}")
    print(f"CUDA available:  {torch.cuda.is_available()}")

    n_channels = X_raw.shape[1]
    n_samples  = X_raw.shape[2]

    eegnet_trainer = EEGNetTrainer(
        n_channels=n_channels,
        n_samples=n_samples,
        n_epochs=150,
        lr=1e-3,
        batch_size=32,
        dropout_rate=0.5,
        patience=20,
        random_state=42
    )

    t0 = time.time()
    eegnet_result = run_loso_cv(X_raw, y, seizure_ids, eegnet_trainer, config=default_config)
    elapsed = time.time() - t0

    agg = eegnet_result['aggregated']
    print(f"\nEEGNet (Raw EEG, LOSO) — trained in {elapsed:.0f}s:")
    print(f"  AUC-ROC:  {agg['roc_auc_mean']:.4f} ± {agg['roc_auc_std']:.4f}")
    print(f"  Sensitivity: {agg['seizure_sensitivity_mean']*100:.1f}%")
    print(f"  FPR/h:    {agg['fpr_per_hour_mean']:.4f}")

except ImportError:
    print("⚠ PyTorch not installed — install with: pip install torch")
    eegnet_result = None
    eegnet_trainer = None

---
## 5. Results Summary Table

In [ ]:
import pandas as pd

all_results = {
    'Baseline (LogReg)':          lr_result,
    'Random Forest':               rf_result,
    'Support Vector Machine':      svm_result,
    'Multilayer Perceptron (ANN)': mlp_result,
}
if eegnet_result is not None:
    all_results['EEGNet (Raw EEG)'] = eegnet_result

rows = []
for name, res in all_results.items():
    agg = res['aggregated']
    rows.append({
        'Model': name,
        'AUC-ROC': f"{agg.get('roc_auc_mean', 0):.4f} ± {agg.get('roc_auc_std', 0):.4f}",
        'PR-AUC': f"{agg.get('pr_auc_mean', 0):.4f} ± {agg.get('pr_auc_std', 0):.4f}",
        'Seizure Sens.': f"{agg.get('seizure_sensitivity_mean', 0)*100:.1f}%",
        'FPR/h': f"{agg.get('fpr_per_hour_mean', 0):.4f}",
        'F1 Score': f"{agg.get('f1_score_mean', 0):.4f}",
    })

df = pd.DataFrame(rows).set_index('Model')
display(df)

---
## 6. Paired Bootstrap Confidence Intervals (ΔAUC)

In [ ]:
# Collect pooled y_true and per-model probabilities across LOSO folds
ref_y_true = lr_result['all_y_true']
model_probs = {
    'Baseline (LogReg)':          lr_result['all_y_prob'],
    'Random Forest':               rf_result['all_y_prob'],
    'Support Vector Machine':      svm_result['all_y_prob'],
    'Multilayer Perceptron (ANN)': mlp_result['all_y_prob'],
}
if eegnet_result is not None:
    model_probs['EEGNet (Raw EEG)'] = eegnet_result['all_y_prob']

bootstrap_results = compare_models_bootstrap(
    y_true=ref_y_true,
    model_probs=model_probs,
    baseline_name='Baseline (LogReg)',
    n_bootstraps=1000,
    random_state=42
)

print(f"{'Model':<35} {'ΔAUC':>8} {'95% CI':>22} {'p-value':>10} {'Sig?':>6}")
print('-' * 85)
for name, res in bootstrap_results.items():
    sig = '✓' if res['is_significant'] else '✗'
    print(f"{name:<35} {res['mean_difference']:>+8.4f} "
          f"[{res['ci_lower']:>+.4f}, {res['ci_upper']:>+.4f}] "
          f"{res['p_value']:>10.4f} {sig:>6}")

---
## 7. Figures

In [ ]:
# Figure 1: ROC + PR curves
eval_results_roc = {
    name: {'y_true': res['all_y_true'], 'y_prob': res['all_y_prob']}
    for name, res in all_results.items()
}
plot_roc_and_pr_curves(
    eval_results=eval_results_roc,
    output_path=FIGURES_DIR / f'{PATIENT_ID}_cp2_roc_pr_curves.png'
)
plt.show()

In [ ]:
# Figure 2: Model comparison bars
agg_results = {n: res['aggregated'] for n, res in all_results.items()}
plot_model_comparison_bars(
    results=agg_results,
    output_path=FIGURES_DIR / f'{PATIENT_ID}_cp2_model_comparison_bars.png'
)
plt.show()

In [ ]:
# Figure 3: Bootstrap CI forest plot
plot_bootstrap_ci(
    bootstrap_results=bootstrap_results,
    output_path=FIGURES_DIR / f'{PATIENT_ID}_cp2_bootstrap_ci.png'
)
plt.show()

In [ ]:
# Figure 4: Random Forest feature importances (fit on all data for stable importances)
rf_final = RandomForestSeizurePredictor(n_estimators=300, random_state=42)
rf_final.fit(X_features, y)

plot_feature_importance(
    importances=rf_final.feature_importances_,
    feature_names=feature_names,
    top_n=20,
    output_path=FIGURES_DIR / f'{PATIENT_ID}_cp2_feature_importance_rf.png'
)
plt.show()

In [ ]:
# Figure 5: Confusion matrices
model_preds_cm = {
    name: {
        'y_true': res['all_y_true'],
        'y_pred': (res['all_y_prob'] >= 0.5).astype(int)
    }
    for name, res in all_results.items()
}
plot_confusion_matrices(
    model_preds=model_preds_cm,
    output_path=FIGURES_DIR / f'{PATIENT_ID}_cp2_confusion_matrices.png'
)
plt.show()

In [ ]:
# Figure 6: EEGNet learning curves
if eegnet_trainer is not None:
    curves = eegnet_trainer.get_learning_curves()
    if curves['train_loss']:
        plot_learning_curves(
            curves=curves,
            output_path=FIGURES_DIR / f'{PATIENT_ID}_cp2_eegnet_learning_curves.png'
        )
        plt.show()

In [ ]:
# Figure 7: Alarm timeline (RF model)
n_windows = len(rf_result['all_y_true'])
window_step_sec = default_config.window_length_sec * (1 - default_config.window_overlap)
time_hours = np.arange(n_windows) * window_step_sec / 3600.0

y_labels = rf_result['all_y_true']
transitions = np.where(np.diff(y_labels.astype(int)) < 0)[0]
seizure_onsets_hours = time_hours[transitions].tolist() if len(transitions) > 0 else [1.5, 3.0]

plot_alarm_timeline(
    time_hours=time_hours,
    alarm_probs=rf_result['all_y_prob'],
    seizure_onsets_hours=seizure_onsets_hours,
    threshold=0.5,
    sop_hours=default_config.sop_min / 60.0,
    sph_hours=default_config.sph_min / 60.0,
    model_name='Random Forest',
    output_path=FIGURES_DIR / f'{PATIENT_ID}_cp2_alarm_timeline.png'
)
plt.show()

---
## 8. Save Results JSON

In [ ]:
def _serialise(obj):
    if isinstance(obj, dict): return {k: _serialise(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)): return [_serialise(v) for v in obj]
    if isinstance(obj, np.integer): return int(obj)
    if isinstance(obj, np.floating): return float(obj)
    if isinstance(obj, np.ndarray): return obj.tolist()
    return obj

summary = {
    'patient': PATIENT_ID,
    'checkpoint': 2,
    'protocol': {
        'cv_method': 'LOSO',
        'sph_min': default_config.sph_min,
        'sop_min': default_config.sop_min,
        'window_length_sec': default_config.window_length_sec,
    },
    'models': {n: _serialise(res['aggregated']) for n, res in all_results.items()},
    'bootstrap_comparison': _serialise(bootstrap_results)
}

out_path = TABLES_DIR / f'{PATIENT_ID}_checkpoint2_summary.json'
with open(out_path, 'w') as f:
    json.dump(summary, f, indent=2)

print(f'Results saved to: {out_path}')
print(json.dumps({n: {k: v for k, v in m.items() if 'mean' in k} for n, m in summary['models'].items()}, indent=2))

---
## 9. Discussion & Error Analysis Roadmap

**Key observations from Checkpoint 2:**
- Do non-linear models improve over the LR baseline on biomarkers? (bootstrap CI)
- How does EEGNet compare to hand-crafted features? (representation comparison)
- Which features drive Random Forest's decision? (feature importances)
- Where does each model fail? (confusion matrices, alarm timeline)

**Checkpoint 3 (Error Analysis) will investigate:**
- Per-seizure false alarm breakdown (which seizures are missed and why?)
- Lead-time analysis (how early before onset does each model fire an alarm?)
- False alarm clustering (are FAs occurring at specific times of day or EEG epochs?)
- Ablation study: Which biomarker subset drives performance?
- Cross-patient generalisation (train on chb01, test on chb03/chb05)?

An honest analysis of failures is more scientifically valuable than inflated accuracy claims.